# Baseline Model — Raidium Challenge 2025

## Table of Contents
1. [Setup & Imports](#setup)
2. [Configuration](#config)
3. [Data Loading & Visualisation](#data)
4. [Model Choice](#model-choice)
5. [Feature Selection & Preprocessing](#feature-selection)
6. [Implementation](#implementation)
   - [Dataset & DataLoader](#dataset)
   - [U-Net Architecture](#unet)
   - [Training Loop](#training)
7. [Evaluation](#evaluation)
8. [Submission Generation](#submission)


---
## 1. Setup & Imports <a id='setup'></a>

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import json
import warnings
from pathlib import Path

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Image I/O ─────────────────────────────────────────────────────────────────
import cv2

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Classical baseline (Watershed) ────────────────────────────────────────────
from skimage.segmentation import watershed
from skimage.filters import sobel, rank
from skimage.morphology import disk
from scipy import ndimage as ndi

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

warnings.filterwarnings('ignore')
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device    : {DEVICE}")

---
## 2. Configuration <a id='config'></a>

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR        = Path("./")                        # root folder with unzipped data
TRAIN_IMG_DIR   = DATA_DIR / "train-images"         # folder of train PNG files
TEST_IMG_DIR    = DATA_DIR / "test-images"           # folder of test  PNG files
LABEL_CSV       = DATA_DIR / "y_train.csv"           # ground-truth labels

# ── Segmentation ──────────────────────────────────────────────────────────────
NUM_CLASSES     = 54   # 55 total (0 = background, 1-54 = organs)
NUM_OUTPUTS     = NUM_CLASSES + 1   # channels out of the network

# ── Training ──────────────────────────────────────────────────────────────────
# We resize to 256x256 for tractable training on modest hardware.
# Predictions are upsampled back to the original resolution before evaluation.
TRAIN_SIZE      = 256      # spatial size fed to the network
VAL_SPLIT       = 200      # first 200 images held out as validation (matches challenge notebook)
BATCH_SIZE      = 4        # increase if you have more VRAM / RAM
NUM_EPOCHS      = 20       # set higher (e.g. 50) for better results
LR              = 3e-4
SEED            = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 3. Data Loading & Visualisation <a id='data'></a>

In [ ]:
def load_images(image_dir: Path) -> np.ndarray:
    """Load PNG images in strict numerical order, returning an (N, H, W) uint8 array."""
    files = sorted(
        Path(image_dir).glob("*.png"),
        key=lambda p: int(p.stem)          # stem = filename without extension
    )
    imgs = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in files]
    return np.stack(imgs, axis=0)


print("Loading training images …")
data_train = load_images(TRAIN_IMG_DIR)
print("Loading test images …")
data_test  = load_images(TEST_IMG_DIR)

# Detect native image resolution from the data (handles both 256 and 512 variants)
IMG_H, IMG_W = data_train.shape[1], data_train.shape[2]

print(f"\nTrain images : {data_train.shape}  (N × H × W)")
print(f"Test  images : {data_test.shape}")
print(f"Native image resolution : {IMG_H} × {IMG_W}")

In [ ]:
# Load labels — note the mandatory transpose!
print("Loading labels …")
labels_train = pd.read_csv(LABEL_CSV, index_col=0).T   # → (N_train, H*W)

print(f"Labels shape : {labels_train.shape}")

# Sanity check: pixel count must match image area
assert labels_train.shape[1] == IMG_H * IMG_W, (
    f"Label columns ({labels_train.shape[1]}) do not match image area "
    f"({IMG_H}×{IMG_W} = {IMG_H * IMG_W})"
)

print(f"Unique label values : {np.unique(labels_train.values[:5])}  (0 = background)")

In [ ]:
def plot_slice_and_mask(image: np.ndarray, mask_flat: np.ndarray, title: str = ""):
    """Side-by-side: raw CT slice vs. coloured segmentation overlay."""
    H, W = image.shape
    mask = mask_flat.reshape(H, W)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("CT Slice")
    axes[0].axis("off")

    axes[1].imshow(image, cmap="gray")
    overlay = np.ma.masked_where(mask == 0, mask)
    axes[1].imshow(overlay, cmap="tab20", alpha=0.6, vmin=1, vmax=NUM_CLASSES)
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")

    if title:
        fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


# Show a few examples
for idx in [0, 50, 100]:
    plot_slice_and_mask(
        data_train[idx],
        labels_train.iloc[idx].values,
        title=f"Training sample #{idx}"
    )

In [ ]:
# ── Label distribution ────────────────────────────────────────────────────────
label_flat = labels_train.values.ravel()
class_ids, class_counts = np.unique(label_flat, return_counts=True)

# Exclude background (class 0) for the plot
fg_ids    = class_ids[class_ids > 0]
fg_counts = class_counts[class_ids > 0]

plt.figure(figsize=(14, 4))
plt.bar(fg_ids, fg_counts, color="steelblue")
plt.xlabel("Organ class ID")
plt.ylabel("Total pixel count")
plt.title("Class distribution in training labels (background excluded)")
plt.tight_layout()
plt.show()

print(f"Background pixels : {class_counts[class_ids == 0][0]:,}")
print(f"Foreground pixels : {fg_counts.sum():,}")
print(f"Foreground fraction : {fg_counts.sum() / label_flat.size:.3%}")

---
## 4. Model Choice <a id='model-choice'></a>

### Why U-Net?

The task is **multi-class semantic segmentation** of abdominal organs in grayscale CT images — exactly the setting U-Net was designed for (Ronneberger *et al.*, 2015). Key arguments:

| Criterion | U-Net |
|---|---|
| Spatial precision | Skip connections preserve fine-grained pixel-level detail lost during downsampling |
| Label efficiency | Originally designed for small medical datasets; generalises well with <1 000 images |
| Established baseline | Standard first choice in every medical image segmentation benchmark |
| Flexibility | Straightforward to swap encoder for a pre-trained backbone (ResNet, EfficientNet) in future iterations |
| Interpretability | Each block has a clear role (encode context → decode localisation) |

### Comparison with challenge starter (Watershed)

The challenge notebook ships an **unsupervised Watershed** baseline that achieves a Dice of ≈ 0.001. Watershed assigns region labels without any knowledge of the ground-truth organ classes — the integer labels it produces are purely topological and have no semantic meaning. A supervised U-Net directly optimises against the ground-truth organ masks, which is the fundamental requirement for meaningful Dice improvement.

### Architecture summary

We use a **lightweight 4-level U-Net** (channel widths 32 → 64 → 128 → 256) that fits on a single GPU with 4 GB VRAM (or on CPU for smaller batch sizes). The output head produces logits over **55 classes** (class 0 = background, classes 1-54 = organs).

---
## 5. Feature Selection & Preprocessing <a id='feature-selection'></a>

### Input features

U-Nets operate on raw pixel values, so the "features" are the pixel intensities themselves. The preprocessing pipeline:

1. **Normalisation**: divide by 255 → float in [0, 1]. CT acquisitions are already stored with consistent windowing by the challenge organisers, so per-image histogram normalisation is deliberately **omitted** for the baseline (to avoid washing out inter-image calibration).
2. **Spatial resize to 256 × 256**: the native 512 × 512 resolution demands ~4× more memory and compute per image. Resizing to 256 × 256 makes the baseline trainable without a data-centre GPU. Predictions are upsampled back with nearest-neighbour interpolation before computing the Dice metric at the original resolution.
3. **Data augmentation (training only)**:
   - Random horizontal flip (p = 0.5)
   - Random vertical flip (p = 0.5)
   - Random brightness jitter (±20 % intensity scale)

### Why not hand-crafted features?

The EDA confirmed that all images are already grayscale and uniformly acquired. There is no strong reason to engineer Gabor filters, HOG, or SLIC superpixels when a convolutional backbone learns task-specific feature detectors end-to-end. Feature engineering is reserved for future ablations if the model under-fits.

---
## 6. Implementation <a id='implementation'></a>

### 6.1 Dataset & DataLoader <a id='dataset'></a>

In [ ]:
class CTSegDataset(Dataset):
    """PyTorch Dataset for CT image segmentation.

    Parameters
    ----------
    images : np.ndarray  (N, H, W)  uint8
    masks  : np.ndarray  (N, H, W)  int  — class indices 0-54
    img_size : int  — spatial size fed to the network
    augment  : bool
    """

    def __init__(
        self,
        images: np.ndarray,
        masks: np.ndarray,
        img_size: int = TRAIN_SIZE,
        augment: bool = False,
    ):
        self.images   = images
        self.masks    = masks
        self.img_size = img_size
        self.augment  = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # --- image ---
        img = self.images[idx].astype(np.float32) / 255.0     # (H, W)
        img = torch.from_numpy(img).unsqueeze(0)              # (1, H, W)

        # --- mask ---
        msk = self.masks[idx].astype(np.int64)                # (H, W)
        msk = torch.from_numpy(msk).unsqueeze(0)              # (1, H, W)

        # --- resize to network input size ---
        img = F.interpolate(
            img.unsqueeze(0), size=self.img_size, mode="bilinear", align_corners=False
        ).squeeze(0)
        msk = F.interpolate(
            msk.float().unsqueeze(0), size=self.img_size, mode="nearest"
        ).squeeze(0).long()

        # --- augmentation ---
        if self.augment:
            if torch.rand(1) > 0.5:
                img = TF.hflip(img)
                msk = TF.hflip(msk)
            if torch.rand(1) > 0.5:
                img = TF.vflip(img)
                msk = TF.vflip(msk)
            # Random brightness jitter
            factor = 0.8 + 0.4 * torch.rand(1).item()   # [0.8, 1.2]
            img = torch.clamp(img * factor, 0.0, 1.0)

        return img, msk.squeeze(0)   # (1,H,W), (H,W)

In [ ]:
# ── Reshape flat labels → (N, H, W) ──────────────────────────────────────────
masks_np = labels_train.values.reshape(-1, IMG_H, IMG_W).astype(np.int64)

# ── Train / validation split (mirrors the challenge notebook convention) ──────
val_imgs   = data_train[:VAL_SPLIT]
val_masks  = masks_np[:VAL_SPLIT]
train_imgs = data_train[VAL_SPLIT:]
train_masks= masks_np[VAL_SPLIT:]

print(f"Train : {train_imgs.shape[0]} images")
print(f"Val   : {val_imgs.shape[0]} images")

# ── Datasets ─────────────────────────────────────────────────────────────────
train_ds = CTSegDataset(train_imgs, train_masks, augment=True)
val_ds   = CTSegDataset(val_imgs,   val_masks,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Quick sanity check on a batch
imgs_b, msks_b = next(iter(train_loader))
print(f"\nSample batch — images: {imgs_b.shape}, masks: {msks_b.shape}")
print(f"Mask unique values (sample): {torch.unique(msks_b).tolist()[:10]} …")

### 6.2 U-Net Architecture <a id='unet'></a>

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    """Two consecutive Conv → BatchNorm → ReLU blocks (same padding)."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    """MaxPool2×2 → DoubleConv."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.pool_conv = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))

    def forward(self, x):
        return self.pool_conv(x)


class Up(nn.Module):
    """Bilinear upsample 2× → DoubleConv (skip connection concatenated)."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        # Handle potential size mismatch (off-by-one from odd resolutions)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


# ─────────────────────────────────────────────────────────────────────────────
# Full U-Net
# ─────────────────────────────────────────────────────────────────────────────

class UNet(nn.Module):
    """Lightweight 4-level U-Net for multi-class CT segmentation.

    Input  : (B, 1, H, W)  — single-channel grayscale
    Output : (B, num_classes, H, W)  — class logits

    Channel widths: 32 → 64 → 128 → 256 → 512 (bottleneck)
    ~7.7 M parameters
    """

    def __init__(self, in_channels: int = 1, num_classes: int = NUM_OUTPUTS, base: int = 32):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, base)         # (B,  32, H,   W  )
        self.enc2 = Down(base,     base * 2)               # (B,  64, H/2, W/2)
        self.enc3 = Down(base * 2, base * 4)               # (B, 128, H/4, W/4)
        self.enc4 = Down(base * 4, base * 8)               # (B, 256, H/8, W/8)

        # Bottleneck
        self.bottleneck = Down(base * 8, base * 16)        # (B, 512, H/16, W/16)

        # Decoder
        self.dec4 = Up(base * 16 + base * 8,  base * 8)   # skip from enc4
        self.dec3 = Up(base *  8 + base * 4,  base * 4)   # skip from enc3
        self.dec2 = Up(base *  4 + base * 2,  base * 2)   # skip from enc2
        self.dec1 = Up(base *  2 + base,       base)      # skip from enc1

        # Output head
        self.head = nn.Conv2d(base, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        bn = self.bottleneck(e4)
        d4 = self.dec4(bn, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.head(d1)


model = UNet().to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters : {total_params:,}")

# Verify forward pass
with torch.no_grad():
    dummy = torch.zeros(1, 1, TRAIN_SIZE, TRAIN_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f"Output shape     : {out.shape}  (B, C, H, W)")

### 6.3 Loss Function

We combine **Cross-Entropy** with **Dice loss**:

- **Cross-Entropy** drives per-pixel classification and handles class imbalance well when combined with class weights.
- **Soft Dice** directly optimises the evaluation metric.

Combined loss = CE + Dice gives stable gradients early in training (CE) and pushes the final Dice score higher (Dice term).

In [ ]:
class DiceLoss(nn.Module):
    """Soft multiclass Dice loss, averaged over foreground classes."""

    def __init__(self, num_classes: int = NUM_OUTPUTS, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits  : (B, C, H, W)  raw logits
        # targets : (B, H, W)     integer class indices
        probs   = F.softmax(logits, dim=1)                    # (B, C, H, W)
        one_hot = F.one_hot(targets, self.num_classes)        # (B, H, W, C)
        one_hot = one_hot.permute(0, 3, 1, 2).float()         # (B, C, H, W)

        # Compute per-class Dice, skip background (class 0)
        intersection = (probs * one_hot)[:, 1:].sum(dim=(0, 2, 3))
        cardinality  = (probs + one_hot)[:, 1:].sum(dim=(0, 2, 3))
        dice_per_cls = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_per_cls.mean()


class CombinedLoss(nn.Module):
    def __init__(self, ce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.ce     = nn.CrossEntropyLoss()
        self.dice   = DiceLoss()
        self.w_ce   = ce_weight
        self.w_dice = dice_weight

    def forward(self, logits, targets):
        return self.w_ce * self.ce(logits, targets) + self.w_dice * self.dice(logits, targets)


criterion = CombinedLoss(ce_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

### 6.4 Training Loop <a id='training'></a>

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    """Run one training or validation epoch. Pass optimizer=None for eval."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    context = torch.enable_grad if is_train else torch.no_grad

    with context():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            logits = model(imgs)
            loss   = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)

    return total_loss / len(loader.dataset)


# ── Training ──────────────────────────────────────────────────────────────────
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(train_loader, model, criterion, optimizer)
    val_loss   = run_epoch(val_loader,   model, criterion, optimizer=None)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")

    print(
        f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}"
        f"  {'✓ saved' if improved else ''}"
    )

print(f"\nBest validation loss : {best_val_loss:.4f}")

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], label="train loss")
plt.plot(history["val_loss"],   label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Combined Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=120)
plt.show()

---
## 7. Evaluation <a id='evaluation'></a>

We reproduce the **exact Dice metric** from the challenge notebook so scores are directly comparable.

In [ ]:
# ── Challenge evaluation functions (copied verbatim from starter notebook) ────

def dice_image(prediction, ground_truth):
    intersection = np.sum(prediction * ground_truth)
    if np.sum(prediction) == 0 and np.sum(ground_truth) == 0:
        return np.nan
    return 2 * intersection / (np.sum(prediction) + np.sum(ground_truth))


def dice_multiclass(prediction, ground_truth):
    dices = []
    for i in range(1, NUM_CLASSES + 1):   # skip background (class 0)
        dices.append(dice_image(prediction == i, ground_truth == i))
    return np.array(dices)


def dice_pandas(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame) -> float:
    """Compute Dice averaged over classes then over images."""
    y_pred_df = y_pred_df.T
    y_true_df = y_true_df.T
    individual_dice = []
    for row_index in range(y_true_df.values.shape[0]):
        dices = dice_multiclass(
            y_true_df.values[row_index].ravel(),
            y_pred_df.values[row_index].ravel(),
        )
        individual_dice.append(dices)
    final    = np.stack(individual_dice)
    cls_dice = np.nanmean(final, axis=0)   # average over images per class
    return float(np.nanmean(cls_dice))     # average over classes

In [ ]:
def predict_dataset(
    images: np.ndarray,
    model: nn.Module,
    device=DEVICE,
    net_size: int = TRAIN_SIZE,
    native_h: int = IMG_H,
    native_w: int = IMG_W,
    batch_size: int = 4,
) -> pd.DataFrame:
    """Run inference on a numpy array of images.

    Returns a DataFrame of shape (N, H*W) matching the challenge label format.
    """
    model.eval()
    all_preds = []

    for start in tqdm(range(0, len(images), batch_size), desc="Predicting"):
        batch_np = images[start : start + batch_size]

        # Preprocess
        batch_t = torch.from_numpy(batch_np.astype(np.float32) / 255.0)
        batch_t = batch_t.unsqueeze(1)   # (B, 1, H, W)
        batch_t = F.interpolate(batch_t, size=net_size, mode="bilinear", align_corners=False)
        batch_t = batch_t.to(device)

        with torch.no_grad():
            logits = model(batch_t)                            # (B, C, net_H, net_W)
            preds  = logits.argmax(dim=1, keepdim=True).float()  # (B, 1, net_H, net_W)

        # Upsample back to native resolution
        preds = F.interpolate(
            preds, size=(native_h, native_w), mode="nearest"
        )                                                      # (B, 1, H_native, W_native)
        preds = preds.squeeze(1).cpu().numpy().astype(np.int32)  # (B, H, W)

        all_preds.append(preds.reshape(len(batch_np), -1))    # (B, H*W)

    return pd.DataFrame(np.concatenate(all_preds, axis=0))

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))

# ── Predict on validation set ─────────────────────────────────────────────────
val_preds_df = predict_dataset(val_imgs, model)

# ── Compute Dice using the challenge metric ────────────────────────────────────
labels_val_df = labels_train.iloc[:VAL_SPLIT]

dice_score = dice_pandas(labels_val_df, val_preds_df)
print(f"\nValidation Dice score (challenge metric) : {dice_score:.6f}")

# For reference, the starter watershed baseline gets ≈ 0.0011
print(f"Challenge watershed baseline              : ~0.0011")

In [ ]:
# ── Per-class Dice breakdown ───────────────────────────────────────────────────
val_preds_T = val_preds_df.T
labels_T    = labels_val_df.T

per_image_dice = []
for i in range(labels_T.values.shape[0]):
    per_image_dice.append(
        dice_multiclass(labels_T.values[i].ravel(), val_preds_T.values[i].ravel())
    )

dice_matrix = np.stack(per_image_dice)                   # (N_val, 54)
cls_dices   = np.nanmean(dice_matrix, axis=0)            # (54,)

plt.figure(figsize=(14, 4))
plt.bar(np.arange(1, NUM_CLASSES + 1), cls_dices, color="steelblue")
plt.xlabel("Organ class ID")
plt.ylabel("Mean Dice")
plt.title("Per-class Dice on Validation Set")
plt.axhline(cls_dices.mean(), color="red", linestyle="--", label=f"Mean = {cls_dices.mean():.4f}")
plt.legend()
plt.tight_layout()
plt.savefig("per_class_dice.png", dpi=120)
plt.show()

# Top-5 and bottom-5
sorted_idx = np.argsort(cls_dices)[::-1]
print("Top-5 classes (highest Dice):")
for i in sorted_idx[:5]:
    print(f"  Class {i+1:2d}: {cls_dices[i]:.4f}")
print("Bottom-5 classes (lowest Dice):")
for i in sorted_idx[-5:]:
    print(f"  Class {i+1:2d}: {cls_dices[i]:.4f}")

In [ ]:
# ── Qualitative inspection ────────────────────────────────────────────────────
val_preds_arr = val_preds_df.values.reshape(VAL_SPLIT, IMG_H, IMG_W)

for idx in [0, 10, 50]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(val_imgs[idx], cmap="gray")
    axes[0].set_title("CT Slice")
    axes[0].axis("off")

    gt = val_masks[idx]
    gt_ov = np.ma.masked_where(gt == 0, gt)
    axes[1].imshow(val_imgs[idx], cmap="gray")
    axes[1].imshow(gt_ov, cmap="tab20", alpha=0.6, vmin=1, vmax=NUM_CLASSES)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    pr = val_preds_arr[idx]
    pr_ov = np.ma.masked_where(pr == 0, pr)
    axes[2].imshow(val_imgs[idx], cmap="gray")
    axes[2].imshow(pr_ov, cmap="tab20", alpha=0.6, vmin=1, vmax=NUM_CLASSES)
    axes[2].set_title("U-Net Prediction")
    axes[2].axis("off")

    fig.suptitle(f"Validation sample #{idx}", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 8. Submission Generation <a id='submission'></a>

In [ ]:
# ── Predict on test set ───────────────────────────────────────────────────────
print("Generating predictions for test set …")
test_preds_df = predict_dataset(data_test, model)
print(f"Test predictions shape : {test_preds_df.shape}")

# ── Format to match submission format ─────────────────────────────────────────
# The challenge expects the transpose (columns = images, rows = pixels)
submission = test_preds_df.T
print(f"Submission shape       : {submission.shape}")

# ── Save ──────────────────────────────────────────────────────────────────────
submission.to_csv("y_pred.csv")
print("Saved → y_pred.csv")

In [ ]:
# ── Quick sanity check on submission ─────────────────────────────────────────
sub_check = pd.read_csv("y_pred.csv", index_col=0)
print(f"Submission shape (loaded) : {sub_check.shape}")
print(f"Unique classes in predictions : {np.unique(sub_check.values)[:10]} …")
print("All looks good — ready to submit!")

---
## Possible Next Steps

This baseline U-Net provides a strong supervised starting point. To push the Dice score further:

1. **Larger input resolution** — train at 512×512 with more VRAM / gradient checkpointing.
2. **Pre-trained encoder** — replace the vanilla encoder with ImageNet-pretrained ResNet-50 or EfficientNet-B4 (single-channel: replicate grey channel to 3 channels or adjust first conv).
3. **nnU-Net** — the automatic configuration framework that wins most medical segmentation benchmarks; try `pip install nnunetv2`.
4. **Test-Time Augmentation (TTA)** — average predictions over flipped/rotated versions of each image.
5. **Class-weighted loss** — weight rare organs higher in the Cross-Entropy term to address class imbalance.
6. **Post-processing** — apply CRF or morphological closing to smooth predictions per organ.
7. **Ensemble** — combine predictions from multiple checkpoints or architectures.